In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import re
from scipy.sparse import csr_matrix, csc_matrix
import pickle

## load data

In [2]:
adata = sc.read_h5ad('../data/TM_FACS/tabula-muris-senis-facs-official-raw-obj.h5ad')

## check outlier cells

In [3]:
adata.obs['n_counts'] = np.sum(adata.X, axis=1)
adata.obs['n_genes'] = np.sum(adata.X > 0, axis=1)
print(adata.obs['n_counts'].min())
print(adata.obs['n_genes'].min())

5002.0
499


In [4]:
sc.pp.filter_genes(adata, min_cells=10)
adata.shape

(110824, 22789)

## convert symbol to id

In [5]:
with open('../data/mouse_name2id.pkl', 'rb') as f:
    gene2id = pickle.load(f)

converted = []
for gene in adata.var_names:
    if gene in gene2id:
        ids = gene2id[gene]
        if len(ids) == 1:
            converted.append(ids[0])
        else:
            converted.append(None)
    else:
        converted.append(None)
adata.var['entrez'] = converted
adata = adata[:, ~pd.isna(adata.var['entrez'])].copy()

In [6]:
# keep genes with max expression across cells
var_df = adata.var.copy()
var_df['mean_exp'] = np.array(np.mean(adata.X, axis=0))[0][:, None]
var_df = var_df.sort_values(by='mean_exp', ascending=False)
sym_to_keep = list(var_df[~var_df['entrez'].duplicated()].index)
adata = adata[:, adata.var.index.isin(sym_to_keep)].copy()
adata.shape

(110824, 22499)

In [7]:
# repalce index
adata.var['symbol'] = adata.var_names
adata.var_names = adata.var['entrez']
del adata.var['entrez']

In [8]:
adata.obs['cell_type'] = adata.obs['cell_ontology_class']

## save

In [9]:
adata.write('../data/TM_FACS/TM_FACS_cnt.h5ad')